# 01 — Data Validation & Cleaning
Load the raw HR dataset, check for missing values, confirm the ordinal 1–4 rating scales are actually in range, fix data types, and save a cleaned version to `data/processed/` for every notebook after this one to use.

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = '../data/raw/Palo_Alto_Networks.csv'
df = pd.read_csv(RAW_PATH)
print('Shape:', df.shape)
df.head()

Shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


## Basic structure check

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   int64
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   str  
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   str  
 14  JobSatisfaction           1470 non-null   int64
 15

## Missing values

In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Columns with missing values:' if len(missing) else 'No missing values found.')
missing

No missing values found.


Series([], dtype: int64)

## Duplicate rows

In [4]:
dupe_count = df.duplicated().sum()
print(f'Duplicate rows: {dupe_count}')
df = df.drop_duplicates()

Duplicate rows: 0


## Validate ordinal (1–4) satisfaction / rating scales
These columns should only contain values 1 through 4. Anything outside that range signals a data entry error.

In [5]:
ordinal_cols = ['EnvironmentSatisfaction', 'JobInvolvement', 'JobSatisfaction',
                'RelationshipSatisfaction', 'WorkLifeBalance']
for c in ordinal_cols:
    bad = df[~df[c].between(1, 4)]
    print(f'{c}: range [{df[c].min()}, {df[c].max()}]  |  out-of-range rows: {len(bad)}')

EnvironmentSatisfaction: range [1, 4]  |  out-of-range rows: 0
JobInvolvement: range [1, 4]  |  out-of-range rows: 0
JobSatisfaction: range [1, 4]  |  out-of-range rows: 0
RelationshipSatisfaction: range [1, 4]  |  out-of-range rows: 0
WorkLifeBalance: range [1, 4]  |  out-of-range rows: 0


## Validate categorical fields
Check that Yes/No and other category columns don't have unexpected typos or values.

In [6]:
cat_checks = ['Attrition', 'BusinessTravel', 'Department', 'Gender',
              'MaritalStatus', 'OverTime']
for c in cat_checks:
    print(f'\n{c}:')
    print(df[c].value_counts())


Attrition:
Attrition
0    1233
1     237
Name: count, dtype: int64

BusinessTravel:
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64

Department:
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

Gender:
Gender
Male      882
Female    588
Name: count, dtype: int64

MaritalStatus:
MaritalStatus
Married     673
Single      470
Divorced    327
Name: count, dtype: int64

OverTime:
OverTime
No     1054
Yes     416
Name: count, dtype: int64


## Normalize Attrition to 0/1 int and standardize dtypes

In [7]:
if df['Attrition'].dtype == object:
    df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})

df['OverTime'] = df['OverTime'].astype(str).str.strip()
df['Department'] = df['Department'].astype(str).str.strip()
print('dtypes standardized.')

dtypes standardized.


## Save cleaned dataset
This is the file every later notebook reads from — the raw CSV in `data/raw/` is never modified.

In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print('Saved:', df.shape, '-> ../data/processed/cleaned_data.csv')

Saved: (1470, 31) -> ../data/processed/cleaned_data.csv
